In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from pathlib import Path

# Adding protein abundance data
This paper has absolute protein abundance data for different E. coli strains in different conditions. The goal of this notebook is to extract protein abundance data and Ribo-Seq data for E. coli grown in glucose M9 (or similar) to check that these transporters are usually expressed


In [3]:
# Data

data_folder = Path('../../../data')
folder = data_folder / 'mori_2021'
fn2 = folder / 'msb20209536-sup-0003-datasetev2.xlsx' # Sample info
fn3 = folder / 'msb20209536-sup-0004-datasetev3.xlsx' # Sample info
fn6 = folder / 'msb20209536-sup-0007-datasetev6.xlsx' # Data for "calibration samples on E. coli in MOPS medium"
fn8 = folder / 'msb20209536-sup-0009-datasetev8.xlsx' # Data for samples listed in fn2
fn9 = folder / 'msb20209536-sup-0010-datasetev9.xlsx' # Data for samples listed in fn3




selected_transporters_fn = data_folder / 'this_project/6_transporterKO' / 'A_selected_transporters.csv'
df_sel = pd.read_csv(selected_transporters_fn)
# df_sel = pd.read_excel(selected_transporters, sheet_name='Table EV1A', skiprows=3, usecols='A:H').iloc[1:]


In [4]:
df6 = pd.read_excel(fn6, sheet_name='EV6-CalibrationSamplesProteins', header = [0,1])
df8 = pd.read_excel(fn8, sheet_name='EV8-AbsoluteMassFractions-1')
df9 = pd.read_excel(fn9, sheet_name='EV9-AbsoluteMassFractions-2')


In [5]:
df6.rename(columns={"Unnamed: 0_level_1":'',	"Unnamed: 1_level_1":'',	"Unnamed: 2_level_1":'',	"Unnamed: 3_level_1":'', "Unnamed: 4_level_1":''}, level = 1, inplace=True)
    

In [6]:
df6.columns = ['_'.join(col) if len(col[1]) else col[0] for col in df6.columns]

In [7]:
xtop1_cols = [x for x in df6.columns if 'xTop' in x]

In [8]:
df6_drop = ['TopPep1_A1-1', 'TopPep1_A1-2', 'TopPep1_A1-3', 'TopPep1_C1',
       'TopPep1_F1-1', 'TopPep1_F1-2', 'TopPep1_F1-3', 'TopPep3_A1-1',
       'TopPep3_A1-2', 'TopPep3_A1-3', 'TopPep3_C1', 'TopPep3_F1-1',
       'TopPep3_F1-2', 'TopPep3_F1-3', 'iBAQ_A1-1', 'iBAQ_A1-2', 'iBAQ_A1-3',
       'iBAQ_C1', 'iBAQ_F1-1', 'iBAQ_F1-2', 'iBAQ_F1-3']
df6.drop(columns=df6_drop, inplace=True)

In [9]:
gene_locus_to_weight = df6.set_index('Gene locus')['Molecular weight (kDa)'].to_dict()

In [10]:
# Convert from mass fractions to number fractions
total_mass = {x: np.sum(df6[x]/df6['Molecular weight (kDa)']) for x in xtop1_cols}
for key, value in total_mass.items():
    df6[key] = (df6[key]/df6['Molecular weight (kDa)'])/value

In [11]:
selected_cols_df8 = ["Lib-24", "Lib-25", "Lib-26", "Lib-27", "Lib-28", "Lib-29", "Lib-30", "Lib-06"] # "Lib-00-A1", "Lib-00-A2", "Lib-00-A3", "Lib-00-B1", "Lib-00-B2", "Lib-00-B3"

selected_cols_df9 = ["A1-1", "A1-2", "A1-3", "C1", "F1-1", "F1-2", "F1-3", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "D6", "D7", "D8", "F4", "F5", "F6", "F7", "F8"]

In [12]:
#Convert df8 to number fractions
df8['Molecular weight (kDa)'] = df8['Gene locus'].map(gene_locus_to_weight)
df9['Molecular weight (kDa)'] = df9['Gene locus'].map(gene_locus_to_weight)

total_mass_8 = {x: np.sum(df8[x]/df8['Molecular weight (kDa)']) for x in selected_cols_df8}
total_mass_9 = {x: np.sum(df9[x]/df9['Molecular weight (kDa)']) for x in selected_cols_df9}
for key, value in total_mass_8.items():
    df8[key] = (df8[key]/df8['Molecular weight (kDa)'])/value
    
for key, value in total_mass_9.items():
    df9[key] = (df9[key]/df9['Molecular weight (kDa)'])/value



In [13]:
df8_all =  ['Lib-01', 'Lib-02', 'Lib-03','Lib-04', 'Lib-05', 'Lib-06', 'Lib-07', 'Lib-08', 'Lib-09', 'Lib-10',
       'Lib-11', 'Lib-12', 'Lib-13', 'Lib-14', 'Lib-15', 'Lib-16', 'Lib-17',
       'Lib-18', 'Lib-19', 'Lib-20', 'Lib-21', 'Lib-22', 'Lib-23', 'Lib-24',
       'Lib-25', 'Lib-26', 'Lib-27', 'Lib-28', 'Lib-29', 'Lib-30']

In [14]:
df9_all = ['A1-1', 'A1-2', 'A1-3', 'C1',
       'F1-1', 'F1-2', 'F1-3', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'D6',
       'D7', 'D8', 'F4', 'F5', 'F6', 'F7', 'F8', 'D1', 'D2', 'D3', 'D4', 'D5',
        'F2', 'F3', 'A2', 'E1', 'E2', 'E3', 'E4', 'H1', 'H5']

In [15]:
# values8 = df8.loc[idx8, selected_cols_df8]

In [16]:
mean_protein_number_fraction = []
std_protein_number_fraction = []
detected_in_all_c_limit = []
detected_in_all = []
ribo_seq = []
for i, row in df_sel.iterrows():
    try:
        idx6 = np.where(df6['Gene locus']==row['Blattner ID'])[0][0]
        idx8 = np.where(df8['Gene locus']==row['Blattner ID'])[0][0]
        idx9 = np.where(df8['Gene locus']==row['Blattner ID'])[0][0]
    except IndexError:
        mean_protein_number_fraction.append(None)
        std_protein_number_fraction.append(None)
        detected_in_all_c_limit.append(None)
        detected_in_all.append(None)
        ribo_seq.append(None)
    else:
        values6 = df6.loc[idx6, xtop1_cols].values
        values8 = df8.loc[idx8, selected_cols_df8].values
        values9 = df9.loc[idx9, selected_cols_df9].values
        values = np.concatenate([values6, values8, values9])
        mean_protein_number_fraction.append(np.mean(values))
        std_protein_number_fraction.append(np.std(values))
        detected_in_all_c_limit.append(np.sum(values>0)/len(values))
        all_values = np.concatenate([df6.loc[idx6, xtop1_cols].values, 
                                     df8.loc[idx8, df8_all].values,
                                     df9.loc[idx9, df9_all].values])
        detected_in_all.append(np.sum(all_values>0)/len(all_values))
            
        ribo_seq.append(df6.loc[idx6, 'Ribosome profiling (Li et al., 2014)'])

In [17]:
len(values)

37

In [18]:
df_sel['Mean protein number fraction'] = mean_protein_number_fraction
df_sel['Std protein number fraction'] = std_protein_number_fraction
df_sel['Detected in fraction of all conditions'] = detected_in_all
df_sel['Detected in fraction of all C-limit conditions'] = detected_in_all_c_limit
df_sel['Ribosome profiling (Li et al., 2014)'] = ribo_seq

In [19]:
df_sel.tail()

,JW ID,Blattner ID,Gene Name,Annotation,Growth rate in glucose + AA medium,Location,Transported metabolites,Metabolite class,Class,Mechanism/specific type,...,Expected direction,TCID,biocyc link,Comment,Paper,Mean protein number fraction,Std protein number fraction,Detected in fraction of all conditions,Detected in fraction of all C-limit conditions,"Ribosome profiling (Li et al., 2014)"
62,JW5735,b4138,dcuA,C4-dicarboxylate antiporter,0.78,Inner membrane,"L-aspartate, fumarate, succinate, malate",Organic acid,C4-dicarboxylate uptake (Dcu) family,Succinate/fumarate antiport,...,Both,NaN,https://biocyc.org/gene?orgid=ECOLI&id=EG11225,DcuA is a C4-dicarboxylate transporter which i...,NaN,0.000103,0.000042,1.0,1.0,30692930.0
63,JW4038,b4077,gltP,glutamate/aspartate:proton symporter,0.86,Inner membrane,"L-aspartate, L-glutamate",Amino acid,dicarboxylate/amino acid:cation symporter (DAA...,Proton symport,...,Import,NaN,NaN,GltP accounts for approximately 60% of the tot...,NaN,0.000000,0.000000,0.0,0.0,6813924.0
64,JW0009,b0010,satP,acetate/succinate:H+ symporter,0.64,Inner membrane,"Acetate, succinate",Organic acid,Acetate Uptake Transporter (AceTr),Proton symport,...,Both,NaN,https://biocyc.org/gene?orgid=ECOLI&id=EG11512,NaN,NaN,0.000000,0.000000,0.0,0.0,889694.0
65,JW2910,b2943,galP,D-galactose transporter,0.84,Inner membrane,"Galactose, D-glucose",Sugar,MFS,Proton symport,...,Import,NaN,https://biocyc.org/gene?orgid=ECOLI&id=EG12148,NaN,NaN,0.000000,0.000000,0.0,0.0,11495660.0
66,JW1652,b1660,punC,predicted transporter,0.76,Inner membrane,Purines,Nucleobase,MFS,Proton symport,...,Import,NaN,https://ecocyc.org/gene?orgid=ECOLI&id=YDHC-MO...,While the punC deletion in general has a negat...,https://pubmed.ncbi.nlm.nih.gov/34413462/,0.000000,0.000000,0.0,0.0,2022080.0


In [20]:
df_sel['log10(Mean protein number fraction)'] = np.log10(df_sel['Mean protein number fraction'])
df_sel['log10(Ribosome profiling)'] = np.log10(df_sel['Ribosome profiling (Li et al., 2014)'])

/Users/snorre/miniconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [21]:
# df_sel.drop(columns=['Location in operon'], inplace=True)
df_sel.replace(-np.inf, np.nan, inplace=True)

In [22]:
fn_new = data_folder / 'this_project' / '6_transporterKO' / 'B_selected_transporters_with_expression_levels.csv'
df_sel.to_csv(fn_new)

# Make transporter metadata table

# Shorter table with only key information

In [25]:
keep_cols = ['JW ID', 'Gene Name', 'Annotation', 'Location', 'Type', 'Expected direction','Metabolite class', 'Transported metabolites']
short_metadata_fn = data_folder / 'this_project' / '6_transporterKO' / 'C_transporters_short_metadata.csv'
df_sel[keep_cols].to_csv(short_metadata_fn, index=False)